In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [ ]:
url = "https://github.com/masterfloss/data/raw/refs/heads/main/fake_new.xlsx"
df = pd.read_excel(url)

df = df.dropna()
display(df.columns)
display(df.head())



In [ ]:
X = df["text"].astype(str)
y = df["label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train= vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Example input size (e.g., TF-IDF features)
input_dim = 5000

model_tf = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(input_dim,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_tf.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# X_train, y_train assumed prepared
model_tf.fit(X_train, y_train, epochs=5, batch_size=32)

loss, acc = model_tf.evaluate(X_test, y_test)
print("TensorFlow MLP Accuracy:", acc)


In [ ]:
from keras.models import Sequential
from keras.layers import Dense

input_dim = 5000

model_keras = Sequential([
    Dense(64, activation='relu', input_shape=(input_dim,)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_keras.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_keras.fit(X_train, y_train, epochs=5, batch_size=32)

loss, acc = model_keras.evaluate(X_test, y_test)
print("Keras MLP Accuracy:", acc)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Convert numpy → torch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)


class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


model_torch = MLP(input_dim=X_train.shape[1])

criterion = nn.BCELoss()
optimizer = optim.Adam(model_torch.parameters(), lr=0.001)

# Training loop
epochs = 20
for epoch in range(epochs):
    model_torch.train()

    outputs = model_torch(X_train_t)
    loss = criterion(outputs, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Evaluation
model_torch.eval()
with torch.no_grad():
    preds = model_torch(X_test_t)
    preds_class = (preds > 0.5).float()

    accuracy = (preds_class.eq(y_test_t).sum() / float(y_test_t.shape[0]))

print("PyTorch MLP Accuracy:", accuracy.item())
